In [ ]:
human_model = cobra.io.load_matlab_model(root_path + 'MammalianSecretoryRecon/MODELS/RECON2_2.mat')
# human_model_2 = cobra.io.load_json_model(local_data_path + 'raw/RECON3D.json')

# two genes in recon2_2 have HGNC:HGNC:### rather than HGNC:###, the following code corrects that issue

# since the two genes are involved in the same three reactions, simply need to rewrite these three reactions
# rather than looping through
genes_to_duplicate = [gene.id for gene in human_model.genes if gene.id.count(':') > 1]
g0, g1 = genes_to_duplicate[0], genes_to_duplicate[1]  
g_i = human_model.genes.get_by_id(g0)
r_i = list(g_i.reactions)
g_c = human_model.genes.get_by_id(g0[5:])

# the following line of code gets rid of both genes in genes to duplicate since they are incolved in the same
# reations
human_model.remove_reactions(r_i, remove_orphans=True)
for r in r_i:
    r0 = r.gene_reaction_rule.replace(g0, g0[5:])
    r.gene_reaction_rule = r0.replace(g1, g1[5:])

# recon2.2 does not have nuclear AMP, adding the transport reaction to model
amp_n = cobra.Metabolite('amp[n]')
amp_n.name = 'AMP(2-)'
amp_n.elements = {'C': 10, 'H': 12, 'N': 5, 'O': 7, 'P': 1}
amp_n.compartment = 'n'
amp_n.charge = -2


amp_transport = cobra.Reaction('AMPtn')
amp_transport.name = 'AMP nuclear transport'
amp_c = human_model.metabolites.get_by_id('amp[c]')
amp_transport.add_metabolites({amp_c: -1, amp_n:1})
amp_transport.lower_bound = -1000
human_model.add_reaction(amp_transport)

# trp_L transport to mitochondria
trp_m = cobra.Metabolite('trp_L[m]')
trp_m.name = 'L-tryptophan'
trp_m.elements = {'C': 11, 'H': 12, 'N': 2, 'O': 2}
trp_m.compartment = 'm'
trp_m.charge = 0

trp_transport = cobra.Reaction('trp_L_MITOCHONDRIAL_MATRIXtn')
trp_transport.name = 'Mitochondrial matrix transport of L-tryptophan'
trp_c = human_model.metabolites.get_by_id('trp_L[c]')
trp_transport.add_metabolites({trp_c: -1, trp_m:1})
trp_transport.lower_bound = -1000
human_model.add_reaction(trp_transport)

aa_ids = ['arg_L[c]', 'asn_L[c]', 'asp_L[c]', 'cys_L[c]', 'glu_L[c]', 'gln_L[c]', 
         'his_L[c]', 'ile_L[c]', 'leu_L[c]', 'met_L[c]', 'phe_L[c]', 'pro_L[c]', 'thr_L[c]', 'trp_L[c]', 
         'tyr_L[c]', 'val_L[c]']
for aa_id in aa_ids:
    aa_c = human_model.metabolites.get_by_id(aa_id)
    aa_x = aa_c.copy()
    aa_x.id = aa_x.id.replace('[c]', '[x]')
    aa_x.compartment = 'x'

    aa_transport = cobra.Reaction(aa_x.id.split('[')[0] + '_PEROXISOMALtn')
    aa_transport.name = aa_x.name + ' transport, peroxisomal'
    aa_transport.add_metabolites({aa_c: -1, aa_x:1})
    aa_transport.lower_bound = -1000
    human_model.add_reaction(aa_transport)


aa_ids = ['ala_L[c]','arg_L[c]','asn_L[c]','asp_L[c]','cys_L[c]','glu_L[c]','gln_L[c]','gly[c]', 'his_L[c]',
 'ile_L[c]','leu_L[c]','met_L[c]','phe_L[c]','pro_L[c]','ser_L[c]','thr_L[c]','trp_L[c]','tyr_L[c]','val_L[c]']
for aa_id in aa_ids:
    aa_c = human_model.metabolites.get_by_id(aa_id)
    aa_n = aa_c.copy()
    aa_n.id = aa_n.id.replace('[c]', '[n]')
    aa_n.compartment = 'n'

    aa_transport = cobra.Reaction(aa_n.id.split('[')[0] + '_NUCLEARtn')
    aa_transport.name = aa_n.name + ' transport, nuclear'
    aa_transport.add_metabolites({aa_c: -1, aa_n:1})
    aa_transport.lower_bound = -1000
    human_model.add_reaction(aa_transport)


# # if metabolite not in compartment or transport reaction not in comaprtemtn



In [ ]:
cobra.io.save_json_model(human_model, local_data_path + 'processed/corrected_recon2_2.json')
# human_model = cobra.io.load_json_model(local_data_path + 'processed/corrected_recon2_2.json')